# Prompt Evaluation Lab

Goal: compare prompt templates using deterministic local scoring before connecting to a GenAI model.

In [ ]:
from pathlib import Path
import pandas as pd

cases = pd.read_csv(Path('../data/prompt_eval_cases.csv'))
cases

In [ ]:
prompt_templates = {
    'short': 'Summarize the input in one sentence: {input_text}',
    'structured': 'Return a structured summary with issue, impact, and next action: {input_text}',
    'sre': 'You are an SRE. Explain incident signal, likely impact, and immediate mitigation: {input_text}',
}

def local_model(prompt: str) -> str:
    return prompt.lower()

def score_response(response: str, expected_keywords: str) -> float:
    keywords = [keyword.strip().lower() for keyword in expected_keywords.split(',')]
    hits = sum(keyword in response for keyword in keywords)
    return hits / len(keywords)


In [ ]:
rows = []
for _, case in cases.iterrows():
    for template_name, template in prompt_templates.items():
        prompt = template.format(input_text=case['input_text'])
        response = local_model(prompt)
        rows.append({
            'case_id': case['case_id'],
            'task': case['task'],
            'template': template_name,
            'score': score_response(response, case['expected_keywords']),
        })

results = pd.DataFrame(rows)
results

In [ ]:
template_summary = results.groupby('template', as_index=False)['score'].mean().sort_values('score', ascending=False)
template_summary

## Interview Talking Points

- Prompt evaluation should use repeatable test cases.
- Compare templates on quality, latency, safety, and cost.
- Local evaluation can happen before expensive model calls.